In [5]:
import zarr
import numpy as np
import os

In [6]:
zarr_path = '/Users/apoliti/minflux_discussion/20250529_grazcydas_LamB1_HMSiR/250529-221048_minflux/'
if not os.path.exists(zarr_path):
    print(f"File {zarr_path} does not exist!")

In [12]:
# Read and keep in memory cache (can be faster, not 100% sure)
base_store = zarr.DirectoryStore(zarr_path)
cache = zarr.LRUStoreCache(store=base_store, max_size=2**28) # for speed reason and reaccess
mfx_data = zarr.open(cache, path = 'mfx')

In [143]:
# Get the MBM data you can recorrect the sample in 
mbm_data = zarr.open(cache, path = 'grd/mbm/points')
mbm_data

<zarr.core.Array '/grd/mbm/points' (14232,) [('gri', '<u4'), ('xyz', '<f8', (3,)), ('tim', '<f4'), ('str', '<f4')]>

In [139]:
tid_mbm = mbm_data['gri'] # id of bead
loc_mbm = mbm_data['xyz'] # localization of bead
tim_mbm = mbm_data['tim'] # time point of measurement
# str not sure what it is
# Also miss the efo and efc....

array([ 61, 123, 185], dtype=uint32)

In [10]:
# Direct load 
mfx_data = zarr.open(zarr_path, mode = 'r', path = 'mfx')

In [12]:
mfx_data


<zarr.core.Array '/mfx' (15872583,) [('vld', '?'), ('fnl', '?'), ('bot', '?'), ('eot', '?'), ('sta', 'u1'), ('tim', '<f8'), ('tid', '<u4'), ('gri', '<u4'), ('thi', 'u1'), ('sqi', 'u1'), ('itr', '<i4'), ('loc', '<f8', (3,)), ('lnc', '<f8', (3,)), ('eco', '<u4'), ('ecc', '<u4'), ('efo', '<f4'), ('efc', '<f4'), ('fbg', '<f4'), ('cfr', '<f2'), ('dcr', '<f2', (2,))] read-only>

In [13]:
# Some utility variables for faster access
tid = mfx_data['tid'] # trace id
tim = mfx_data['tim'] # Time of iteration 
vld = mfx_data['vld'] # valid tag for each localization
loc = mfx_data['loc'] # localization coordinates corrected for MBM
itr = mfx_data['itr'] # iteration of trace
efc = mfx_data['efc'] # effective frequency center
efo = mfx_data['efo'] # effective frequency outer


In [14]:
# Shape of objects
print(f"vld has shape {vld.shape}, (n_localization,)")
print(f"tid has shape {tid.shape}, (n_localization,)")
print(f"loc has shape {loc.shape} , (n_localization, X, Y, Z)")
print(f"efc has shape {efc.shape} , (n_localization,)")
print(f"itr has shape {itr.shape} , (n_localization,)")

vld has shape (15872583,), (n_localization,)
tid has shape (15872583,), (n_localization,)
loc has shape (15872583, 3) , (n_localization, X, Y, Z)
efc has shape (15872583,) , (n_localization,)
itr has shape (15872583,) , (n_localization,)


In [17]:
vld

array([ True, False, False, ..., False, False, False], shape=(15872583,))

In [23]:
# Get only valid localizations
loc_vld = loc[vld]
tid_vld = tid[vld]
itr_vld = itr[vld]
efo_vld = efo[vld]
tim_vld = tim[vld]
print(f"itr_vld has shape {itr_vld.shape} , (n_localization,)")

itr_vld has shape (452985,) , (n_localization,)


In [19]:
# Get only the traces that reached the last itr
lst_itr_value = np.max(itr_vld)
print(f'lst_itr_value {lst_itr_value}')  

lst_itr_value 7


In [20]:
itr_vld == lst_itr_value

array([False, False, False, ..., False, False, False], shape=(452985,))

In [24]:
# Only the last iteration
loc_vld_final = loc_vld[itr_vld == lst_itr_value]
tid_vld_final = tid_vld[itr_vld == lst_itr_value]
itr_vld_final = itr_vld[itr_vld == lst_itr_value]
efo_vld_final = efo_vld[itr_vld == lst_itr_value]
tim_vld_final = tim_vld[itr_vld == lst_itr_value]

In [31]:
tid_vld_final

array([    897,     897,     898, ..., 1992127, 1992127, 1992127],
      shape=(59863,), dtype=uint32)

In [90]:
# All the iterations for those that reach the final iteration
tids_to_mask = np.unique(tid_vld[itr_vld == 7])
final_mask = np.isin(tid_vld, tids_to_mask)
loc_vld_final_all = loc_vld[final_mask]
tid_vld_final_all = tid_vld[final_mask]
itr_vld_final_all = itr_vld[final_mask]

In [36]:
import matplotlib.pyplot as plt
import numpy as np

u_tid, counts = np.unique(tid_vld_final, return_counts = True)
plt.hist(counts[counts < 40], bins=10, color='skyblue', edgecolor='black')

# Adding labels for clarity
plt.title('Histogram of Counts')
plt.xlabel('Value')

plt.ylabel('Frequency')

plt.show()
#u_tid[counts > 2]
#loc_vld_final[np.isin(tid_vld_final, u_tid[counts > 2])]

In [98]:
import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default = 'notebook'

In [109]:
loc_to_plot

array([[-1.41437292e-05,  1.94643899e-05, -3.01229592e-07]])

In [140]:
# Create the 3D scatter plot
z_fact = 0.7
min_loc_per_trace = 3
u_tid, counts = np.unique(tid_vld, return_counts = True)
loc_to_plot = loc_vld_final[np.isin(tid_vld_final, u_tid[counts>=min_loc_per_trace])]
x_coords = loc_to_plot[:, 0]
y_coords = loc_to_plot[:, 1]
z_coords = loc_to_plot[:, 2]*z_fact
fig = go.Figure(data=[go.Scatter3d(
    x=x_coords,
    y=y_coords,
    z=z_coords,
    mode='markers',
    marker=dict(
        size=1,
        color=z_coords,                # set color to an array/list of desired values
        colorscale='Viridis',   # choose a colorscale
        opacity=0.8
    )
)])
# Update the layout for a better presentation
fig.update_layout(
    title='3D Scatter Plot of Points',
    scene=dict(
        xaxis_title='X Axis',
        yaxis_title='Y Axis',
        zaxis_title='Z Axis', 
        aspectmode='data'
    ),
    margin=dict(r=20, b=10, l=10, t=40)
)

# Show the interactive plot
print("\nDisplaying interactive 3D plot...")
fig.show()


Displaying interactive 3D plot...


In [124]:
import plotly.express as px
# --- 2. Create the Histogram ---
# The 'Value' column is passed to the x-axis for the histogram.
fig = px.histogram(
    {'efo':efo_vld_final},
    x="efo",
    nbins=30,  # Specify the number of bins (or let Plotly choose)
    title="EFO"
)

# --- 3. Display the Plot ---
# This command is what renders the interactive plot in your notebook shell.
fig.show()

In [120]:
# compute the std for each trace id and remove those localization that are out of range
# later we can do DBScan on each of those localizations

array([77243.016, 46543.87 , 45553.574, ..., 75262.43 , 50505.05 ,
       65359.477], shape=(59863,), dtype=float32)